In [1]:
# -------------------------------------------------------------------------
# 第一步：安裝必要的函式庫
# -------------------------------------------------------------------------
!pip install transformers datasets pandas accelerate sentencepiece -U
print("✅ 環境安裝完成")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 91.2/91.2 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 511.6/511.6 kB 20.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.4/12.4 MB 95.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.7/47.7 MB 17.5 MB/s eta 0:00:00
  Attempting uninstall: pyarrow
    Found existing installation: pyarrow 18.1.0
    Uninstalling pyarrow-18.1.0:
      Successfully uninstalled pyarrow-18.1.0
  Attempting uninstall: pandas
    Found existing installation: pandas 2.2.2
    Uninstalling pandas-2.2.2:
      Successfully uninstalled pandas-2.2.2
  Attempting uninstall: datasets
    Found existing installation: datasets 4.0.0
    Uninstalling datasets-4.0.0:
      Successfully uninstalled datasets-4.0.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires pandas=

In [2]:
!pip install "numpy<2.0"

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 52.8 MB/s eta 0:00:00
  Attempting uninstall: numpy
    Found existing installation: numpy 2.0.2
    Uninstalling numpy-2.0.2:
      Successfully uninstalled numpy-2.0.2
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires pandas==2.2.2, but you have pandas 2.3.3 which is incompatible.
pytensor 2.35.1 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
opencv-python 4.12.0.88 requires numpy<2.3.0,>=2; python_version >= "3.9", but you have numpy 1.26.4 which is incompatible.
jaxlib 0.7.2 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
jax 0.7.2 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
shap 0.50.0 requires numpy>=2, but you have numpy 1.2

In [1]:
# -------------------------------------------------------------------------
# 第二步：匯入套件並讀取特定 Topic 資料
# -------------------------------------------------------------------------
import pandas as pd
from datasets import Dataset
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, TrainingArguments, Trainer, DataCollatorForLanguageModeling

# ================= 設定區 =================
FILE_PATH = "JYP_2018_Topic4_songs.csv" # 您的檔案名稱
TARGET_TOPIC_ID = 3                     # 您的目標 Topic (例如 2)
MODEL_NAME = "skt/kogpt2-base-v2"       # 預訓練韓文模型
# =========================================

# 1. 讀取 CSV
try:
    df = pd.read_csv(FILE_PATH)
    print(f"📂 成功讀取檔案，共有 {len(df)} 筆資料。")
except FileNotFoundError:
    print("❌ 找不到檔案！請確認 CSV 是否已上傳到 Colab 左側的檔案區。")
    raise

# 2. 篩選資料 (Filter)
# 只保留 topic_id 為目標數字，且歌詞不是空的
filtered_df = df[(df['Dominant_Topic_index'] == TARGET_TOPIC_ID) & (df['lyrics'].notna())]
data_count = len(filtered_df)

print(f"📊 篩選結果：Topic {TARGET_TOPIC_ID} 共有 {data_count} 首歌。")

if data_count == 0:
    raise ValueError(f"❌ 錯誤：Topic {TARGET_TOPIC_ID} 沒有任何歌曲，請檢查數字是否正確。")

# 3. 轉換為 HuggingFace Dataset 格式
# 我們將所有歌詞轉換為字串列表
dataset_dict = {
    'text': filtered_df['lyrics'].astype(str).tolist()
}
dataset = Dataset.from_dict(dataset_dict)

print("\n--- 歌詞樣本預覽 ---")
print(dataset[0]['text'][:100] + "...")

📂 成功讀取檔案，共有 49 筆資料。
📊 篩選結果：Topic 3 共有 37 首歌。

--- 歌詞樣本預覽 ---
총량의 법칙

Grrr 정신은 지저분해
난폭한 개마냥 누구 앞에
서 있든지 상관 않고 짖어 분해
이해 못 해주는 이해심은 처분해
나를 향한 시선들이 좋든 안 좋든
다 따갑게 느껴지...


In [2]:
# -------------------------------------------------------------------------
# 第三步：載入 Tokenizer 並處理資料 (修正版)
# -------------------------------------------------------------------------
print("⏳ 正在下載並載入模型 (skt/kogpt2-base-v2)...")

# 1. 載入分詞器
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
# KoGPT2 沒有預設的 pad_token，我們將其設為 eos_token
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# 2. 定義分詞函數 (加入更嚴格的截斷與長度控制)
def tokenize_function(examples):
    return tokenizer(
        examples["text"],
        padding="max_length",
        truncation=True,
        max_length=512,       # 嚴格限制長度
        return_token_type_ids=False # GPT2 不需要這個，有時候會導致問題
    )

# 3. 應用分詞
tokenized_datasets = dataset.map(tokenize_function, batched=True)

# 4. 準備訓練所需的欄位
tokenized_datasets.set_format(type='torch', columns=['input_ids', 'attention_mask'])

# 5. 【關鍵修正】檢查是否有壞掉的數據 (Token ID 超出範圍)
# 雖然我們後面會 resize 模型，但檢查一下比較保險
vocab_size_limit = tokenizer.vocab_size
def validate_data(example):
    # 確保所有 ID 都在合理範圍內 (預留一點緩衝空間)
    return all(id < 60000 for id in example['input_ids'])

# 過濾掉異常數據
original_len = len(tokenized_datasets)
tokenized_datasets = tokenized_datasets.filter(validate_data)
if len(tokenized_datasets) < original_len:
    print(f"⚠️ 自動過濾了 {original_len - len(tokenized_datasets)} 筆異常資料")

train_dataset = tokenized_datasets
print("✅ 資料處理完成！")

⏳ 正在下載並載入模型 (skt/kogpt2-base-v2)...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Map:   0%|          | 0/37 [00:00<?, ? examples/s]

Filter:   0%|          | 0/37 [00:00<?, ? examples/s]

✅ 資料處理完成！


In [3]:
# -------------------------------------------------------------------------
# 第四步：設定訓練參數並開始微調 (修正版)
# -------------------------------------------------------------------------

# 1. 載入模型
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME)

# 【關鍵修正】強制調整模型詞彙表大小以匹配 Tokenizer
# 這行能解決大部分 "device-side assert" 錯誤
model.resize_token_embeddings(len(tokenizer))

# 2. 設定資料收集器
data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False
)

# 3. 設定訓練參數
training_args = TrainingArguments(
    output_dir="./fine_tuned_kpop_lyrics",
    overwrite_output_dir=True,
    num_train_epochs=10,
    per_device_train_batch_size=4, # 如果還是記憶體不足，可改為 2
    gradient_accumulation_steps=1, # 搭配 batch size=2 時可設為 2
    save_steps=50,
    logging_steps=10,
    learning_rate=5e-5,
    prediction_loss_only=True,
    report_to="none",
    fp16=True, # 【建議】開啟混合精度訓練，通常能減少記憶體壓力並避免某些錯誤
)

# 4. 建立 Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    data_collator=data_collator,
)

# 5. 開始訓練
print("🚀 開始訓練模型 (Fine-tuning)... 這可能需要幾分鐘")
trainer.train()

# 6. 儲存模型
trainer.save_model("./final_kpop_model")
print("🎉 訓練完成！模型已儲存。")

pytorch_model.bin:   0%|          | 0.00/513M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/513M [00:00<?, ?B/s]

The new embeddings will be initialized from a multivariate normal distribution that has old embeddings' mean and covariance. As described in this article: https://nlp.stanford.edu/~johnhew/vocab-expansion.html. To disable this, use `mean_resizing=False`


🚀 開始訓練模型 (Fine-tuning)... 這可能需要幾分鐘


`loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


Step,Training Loss
10,3.458900
20,2.654200
30,2.315800
40,1.945500
50,1.760100
60,1.526300
70,1.329900
80,1.215600
90,1.139400
100,1.079100


🎉 訓練完成！模型已儲存。


In [ ]:
# =================================================================
# 🎵 批量生成 10 首歌並匯出 Excel (Batch Generation) - 修正版
# =================================================================
import pandas as pd
import torch
from datetime import datetime
try:
    from google.colab import files # 用於自動下載檔案
except ImportError:
    pass

# ---------------------------------------------------------
# 🛠️ 修正部分：在此處定義 device 並確保模型在正確的裝置上
# ---------------------------------------------------------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"🖥️ 使用裝置: {device}")
model.to(device)
# ---------------------------------------------------------

# 1. 準備儲存容器
generated_songs = []
NUM_SONGS = 10 # 您要生成的數量

print(f"🚀 開始生成 {NUM_SONGS} 首新歌... 請稍候")
print("-" * 50)

# 2. 定義生成函數 (確保每次有些微不同)
def generate_one_song():
    # 取得開始信號
    start_token = tokenizer.bos_token_id if tokenizer.bos_token_id else tokenizer.eos_token_id

    # 這裡原本報錯，因為 device 沒定義，現在已經修復
    input_ids = torch.tensor([[start_token]], device=device)

    with torch.no_grad():
        output = model.generate(
            input_ids,
            max_length=200,           # 歌詞長度 (可自行調整)
            temperature=1.0,          # 創意度 (1.0 標準, 1.2 更瘋狂)
            top_k=50,                 # 選詞範圍
            top_p=0.95,               # 核採樣
            repetition_penalty=1.2,   # 避免重複
            do_sample=True,           # 隨機採樣 (確保每首不一樣)
            pad_token_id=tokenizer.eos_token_id
        )
    return tokenizer.decode(output[0], skip_special_tokens=True)

# 3. 執行迴圈
for i in range(NUM_SONGS):
    print(f"[{i+1}/{NUM_SONGS}] 正在創作第 {i+1} 首...")

    # 生成歌詞
    lyrics = generate_one_song()

    # 存入列表
    generated_songs.append({
        "Song_ID": i + 1,
        "Topic_Group": "Topic 2 (Generated)", # 您可以改成對應的 Topic
        "Generated_Time": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
        "Lyrics": lyrics
    })

print("-" * 50)
print("✅ 生成完成！正在製作 Excel 檔案...")

# 4. 轉換為 DataFrame 並存檔
df_result = pd.read_json(pd.DataFrame(generated_songs).to_json()) # 確保格式乾淨
file_name = "10_Generated_KPop_Songs4群.xlsx"

# 匯出 Excel
df_result.to_excel(file_name, index=False, engine='openpyxl')
print(f"📂 檔案已儲存為: {file_name}")

# 5. 自動下載 (僅限 Colab)
try:
    files.download(file_name)
    print("⬇️ 下載請求已發送。")
except Exception as e:
    print("⚠️ 無法自動下載，請從左側檔案總管手動下載。")

🖥️ 使用裝置: cuda
🚀 開始生成 10 首新歌... 請稍候
--------------------------------------------------
[1/10] 正在創作第 1 首...
[2/10] 正在創作第 2 首...
[3/10] 正在創作第 3 首...
[4/10] 正在創作第 4 首...


In [3]:
import pandas as pd
import torch
import numpy as np
from transformers import AutoTokenizer, AutoModelForCausalLM
import os

try:
    from google.colab import files
except ImportError:
    pass

# ==========================================
# 1. 設定與重新載入模型 (Fix NameError)
# ==========================================
INPUT_FILE = "10_Generated_KPop_Songs8群.xlsx"  # 請確認檔名一致
OUTPUT_FILE = "10_Generated_KPop_Songs_with_PPL8.xlsx"

# 決定要讀取哪個模型
# 如果你的微調模型還在 (沒有被 Colab 回收)，我們優先讀取微調後的
# 如果微調模型不見了，就讀取原始韓文模型 (分數會比較高，但能跑)
fine_tuned_path = "./final_kpop_model"
base_model_name = "skt/kogpt2-base-v2"

if os.path.exists(fine_tuned_path):
    print(f"📂 發現微調模型，從 {fine_tuned_path} 載入...")
    model_name_or_path = fine_tuned_path
else:
    print(f"⚠️ 找不到微調模型資料夾 (可能因重啟被刪除)，改為載入原始模型 {base_model_name}...")
    model_name_or_path = base_model_name

print("⏳ 正在載入模型與分詞器...")
tokenizer = AutoTokenizer.from_pretrained(model_name_or_path)
model = AutoModelForCausalLM.from_pretrained(model_name_or_path)

# 設定 pad_token (避免報錯)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# 搬移到 GPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
model.eval()
print("✅ 模型載入完成！")

# ==========================================
# 2. 定義 PPL 計算函數
# ==========================================
def calculate_perplexity(text):
    if not isinstance(text, str) or len(text.strip()) == 0:
        return float('nan')

    # 編碼
    encodings = tokenizer(text, return_tensors="pt")
    input_ids = encodings.input_ids.to(device)

    # 計算 Loss (Log-Likelihood)
    with torch.no_grad():
        outputs = model(input_ids, labels=input_ids)
        loss = outputs.loss

    # 轉換為 PPL
    return torch.exp(loss).item()

# ==========================================
# 3. 讀取 Excel 並計算
# ==========================================
print(f"\n📂 正在讀取檔案: {INPUT_FILE} ...")
try:
    df = pd.read_excel(INPUT_FILE)
    print(f"✅ 成功載入 {len(df)} 筆歌詞資料。")
except FileNotFoundError:
    print("❌ 找不到檔案！請確認檔名是否正確，或檔案是否已上傳到 Colab。")
    raise

print("\n🚀 開始計算 Perplexity...")
print("-" * 30)

# 使用 apply 批量計算
df['Perplexity'] = df['Lyrics'].apply(lambda x: calculate_perplexity(x))

# ==========================================
# 4. 統計與存檔
# ==========================================
valid_ppl = df['Perplexity'].dropna()

print("\n📊 [統計摘要 Statistics]")
print(f"樣本數 (N): {len(valid_ppl)}")
print(f"平均值 (Mean): {valid_ppl.mean():.2f}")
print(f"標準差 (Std Dev): {valid_ppl.std():.2f}")
print(f"最小值 (Min): {valid_ppl.min():.2f}")
print(f"最大值 (Max): {valid_ppl.max():.2f}")

# 排序並存檔
df_sorted = df.sort_values(by="Perplexity", ascending=True)
df_sorted.to_excel(OUTPUT_FILE, index=False, engine='openpyxl')
print(f"\n💾 已儲存新檔案: {OUTPUT_FILE}")

try:
    files.download(OUTPUT_FILE)
    print("⬇️ 下載請求已發送。")
except Exception:
    pass

⚠️ 找不到微調模型資料夾 (可能因重啟被刪除)，改為載入原始模型 skt/kogpt2-base-v2...
⏳ 正在載入模型與分詞器...
✅ 模型載入完成！

📂 正在讀取檔案: 10_Generated_KPop_Songs8群.xlsx ...
✅ 成功載入 10 筆歌詞資料。

🚀 開始計算 Perplexity...
------------------------------

📊 [統計摘要 Statistics]
樣本數 (N): 10
平均值 (Mean): 48.24
標準差 (Std Dev): 12.69
最小值 (Min): 30.97
最大值 (Max): 72.71

💾 已儲存新檔案: 10_Generated_KPop_Songs_with_PPL8.xlsx


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

⬇️ 下載請求已發送。
